# 2. Análise: seleção de variáveis e agrupamento (RMSP)

Etapas: triagem estatística das variáveis → correlação de Pearson e critérios conceituais → conversão em taxas → verificação de redundância → normalização Min-Max → K-Means → testes de estabilidade (sementes e Ward) → caracterização dos grupos → variável suplementar (favelas).

Recorte: 39 municípios da Região Metropolitana de São Paulo. Semente fixa (`random_state=42`) para reprodutibilidade.

In [ ]:
# Library versions used in this run (copy them into requirements.txt to pin the environment)
import importlib
for lib in ["pandas", "numpy", "sklearn", "scipy", "matplotlib", "seaborn", "plotly", "geopandas"]:
    try:
        print(f"{lib}=={importlib.import_module(lib).__version__}")
    except ImportError:
        print(f"{lib}: not installed")

In [ ]:
# 1. LOAD CONSOLIDATED DATA FROM GITHUB

import pandas as pd

# Processed files generated by 1_BD_consolidacao.ipynb (1_DATA/processed)
path = "https://raw.githubusercontent.com/julianevesa/rmsp-habitacao-censo2022/main/1_DATA/processed/"

# Load the data directly from GitHub
df = pd.read_csv(path + "base_final_SP.csv", sep=";")

Spatial Scope: São Paulo Metropolitan Region (RMSP)

Only the 39 municipalities of the RMSP are kept. All subsequent steps (rates, Min-Max normalization and K-Means) are computed only for these municipalities.

In [ ]:
import unicodedata

SCOPE = "RMSP"  # This repository refers only to the São Paulo Metropolitan Region (39 municipalities)

RMSP = [
    "Arujá", "Barueri", "Biritiba Mirim", "Caieiras", "Cajamar", "Carapicuíba",
    "Cotia", "Diadema", "Embu das Artes", "Embu-Guaçu", "Ferraz de Vasconcelos",
    "Francisco Morato", "Franco da Rocha", "Guararema", "Guarulhos",
    "Itapecerica da Serra", "Itapevi", "Itaquaquecetuba", "Jandira", "Juquitiba",
    "Mairiporã", "Mauá", "Mogi das Cruzes", "Osasco", "Pirapora do Bom Jesus",
    "Poá", "Ribeirão Pires", "Rio Grande da Serra", "Salesópolis", "Santa Isabel",
    "Santana de Parnaíba", "Santo André", "São Bernardo do Campo",
    "São Caetano do Sul", "São Lourenço da Serra", "São Paulo", "Suzano",
    "Taboão da Serra", "Vargem Grande Paulista",
]

def norm(name):
    s = unicodedata.normalize("NFD", str(name)).encode("ascii", "ignore").decode()
    return s.lower().replace("-", " ").strip()

rmsp_norm = {norm(n) for n in RMSP}
df = df[df["NM_MUN"].map(norm).isin(rmsp_norm)].reset_index(drop=True)
found = {norm(n) for n in df["NM_MUN"]}
missing = [n for n in RMSP if norm(n) not in found]
print(f"RMSP municipalities found: {len(df)} of 39")
if missing:
    print("Not found (check spelling in NM_MUN):", missing)
assert len(df) == 39, "The RMSP must have 39 municipalities."

The first step was to open the 2022 Census dataset and examine all available numerical variables at once, without making any initial selections or applying filters. To do this, we ran a simple statistical analysis on all numerical columns to determine the mean, minimum, and maximum values ​​for each, while also checking for missing or zeroed-out data. The goal was to gain an overview of the data and prepare for subsequent steps—using mathematics and correlation tests—to decide which variables truly help explain housing conditions in São Paulo in the context of SDG 11 (Target 1), and which variables could and should be discarded to ensure an accurate analysis free from noise or bias.

In [ ]:
import numpy as np
import pandas as pd

# 1. Filters the dataframe, keeping only the numeric columns
df_num = df.select_dtypes(include=[np.number])

# 2. Creates the basic statistics table
estatistica = df_num.describe().T

estatistica["qtd_nulos"] = df_num.isnull().sum()
estatistica["qtd_zeros"] = (df_num == 0).sum()

display(
    estatistica[
        ["count", "mean", "min", "max", "qtd_nulos", "qtd_zeros"]
    ].round(2)
)

Initial Correlation Analysis

Since the original 2022 Census dataset comprises 644 variables, it is necessary to understand how this information behaves when cross-referenced before making any decisions or creating new rates. The goal of applying a Pearson correlation test at this stage is to identify data points that are redundant or act as a "mirror" of one another.

Performing this check before selecting the final variables is crucial for two reasons: first, the IBGE often repeats the same counts across different tables, and there are variables that present cross-tabulated data—meaning they combine two or more pieces of information in a single column (such as the number of "house" type dwellings where the head of the household is female). Second, if identical variables or those containing such cross-tabulated data are included in the artificial intelligence model, the algorithm will assign double weight to the same subject, thereby skewing the final result. Through this mathematical analysis, we can map all redundancies in a spreadsheet format, allowing us to confidently determine in the next step which columns to keep and which to discard.

In [ ]:
import numpy as np
import pandas as pd

import numpy as np
import pandas as pd

# 1. Calculates the correlation matrix
correlation_matrix = df_num.corr(method="pearson")

# 2. Converts the matrix into a list of pairs
df_pairs = (
    correlation_matrix.where(
        np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
    )
    .stack()
    .reset_index()
)
df_pairs.columns = ["Variable A", "Variable B", "Correlation"]

# 3. Filters strong correlations
threshold = 0.85
df_screening = df_pairs[df_pairs["Correlation"].abs() >= threshold].copy()

# 4. Defines the Relationship Type
df_screening["Relationship Type"] = np.where(
    df_screening["Correlation"] >= threshold,
    "Direct (Same Direction)",
    "Inverse (Mirror)",
)

# 5. Saves the final dataset locally
df_screening = df_screening.sort_values(by="Correlation", ascending=False).round(
    2
)
df_screening.to_csv(
    "initial_variable_correlation_analysis.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False,
)

Definition and Selection of Final Variables

Based on the correlation matrix generated from the initial 643 variables, we made a final selection of the raw columns that were truly relevant to the study. The selection criterion involved eliminating data points that showed perfect correlation (absolute multicollinearity with a value of 1.00) to avoid artificially inflating the weight of any specific topic. For instance, regarding variables V00238 and V00316 (which both measured bathroom facilities but appeared in different IBGE tables), we chose to retain only V00316—which focuses on the total absence of bathroom facilities—and discarded the other. Ultimately, we compiled a dataset comprising raw variables related to basic infrastructure (water, sewage, waste disposal), housing precariousness (tenement housing, dilapidated structures, makeshift dwellings), and social aspects (residents and children), thereby ensuring a clean dataset free of redundant information.

3.4 Justification and Consolidation of Selected Variables

Following automated Pearson correlation tests and a manual qualitative screening process, the dataset was filtered and reduced to the definitive variables comprising the study. This selection was guided by three fundamental criteria: the elimination of statistical redundancies, alignment with the absolute deprivation perspective advocated by the UN for achieving SDG Target 11.1, and a preference for "pure" variables—those representing a single, isolated piece of information.

Discarding variables that combined data points—such as household counts linked specifically to the sex or race of the head of the household—was essential to ensure the model's neutrality. By selecting columns that strictly measure structural or social deficits in isolation (such as the absolute total of households lacking a bathroom or the total number of children), secondary demographic factors are prevented from confounding the measurement of urban precariousness, thereby isolating the phenomenon of interest.

Similarly, variables indicating the presence of adequate infrastructure—such as regular access to the sewage network or "households with exclusive-use bathrooms"—were discarded in favor of those capturing the total absence of service or the reliance on substandard solutions. Statistically, these adequacy variables showed a strong negative correlation with vulnerability variables; conceptually, focusing on absolute deprivation ensures that the artificial intelligence model identifies municipalities based on critical deficits rather than established service coverage. The organization and methodological rationale for the final block of raw variables are detailed below:

1. Population and Territorial Denominators:

V00001 (Occupied Permanent Private Households - DPPO): Retained as the official and exclusive denominator for calculating all urban physical infrastructure rates (water, sewage, waste disposal, bathrooms, and permanent housing types), serving as a comparative baseline to isolate the effect of municipality size.

v0007 (Total Occupied Private Households): Used as an expanded denominator, comprising the sum of permanent and improvised households (DPPO + DPIO); it is applied specifically to calculate indicators of absolute social and housing vulnerability.

2. Social and Demographic Vulnerability Profile:

V00005 and V00008 (Residents and Children aged 0–9 in DPPO): Selected to measure population density and the presence of early childhood populations in permanent structures.

V00006 and V00009 (Residents and Children aged 0–9 in DPIO): Retained to highlight the most critical social scenario, accounting for the number of people and children living in conditions of extreme improvisation.

3. Territorial Vulnerability and Urban Segregation:

N_POP_DPPO (Resident population in *favelas* and urban communities) — NOT INCLUDED in the clustering: precariousness is characterized here by household conditions (access to water, sewage and waste services and adequacy of the dwelling structure), not by the territorial classification of the settlement, which by itself does not describe these conditions. Since IBGE's definition of favelas and urban communities already considers deficiencies in services and infrastructure, including it would overlap information already represented by the other variables and increase its weight in the clustering. This variable comes from a separate IBGE dataset and is analyzed later only as a supplementary variable, overlaid on the clusters.

4. Housing Typologies and Physical Precarity:

V00050, V00051, and V00052 (Tenements, Shacks, and Dilapidated Structures): Retained to represent inadequacies found within the so-called permanent urban fabric (DPPO).

V00053 to V00058 (Tents, Commercial Premises, Natural Shelters, Public Spaces, Non-Residential Structures, and Vehicles): This block of variables regarding improvised dwellings (DPIO) was kept in its entirety for subsequent grouping into a single indicator of extreme housing vulnerability, ensuring that no form of homelessness or substandard housing was omitted.

5. Water Supply Deprivation:

V00113 to V00118 (Alternative Water Sources): Retained to record the use of water without guaranteed prior treatment (shallow wells, springs, water trucks, rainwater, rivers, and reservoirs).

V00200, V00201, and V00464 (External Piping and Lack of Connection): Selected to measure the deficit in internal distribution and the absence of a physical connection to the public utility network.

6. Sanitation and Sewage Deprivation:

V00236 and V00237 (Communal Bathroom and Rudimentary Toilet): Retained as they represent shared usage or structures lacking minimum hygiene standards.

V00312 to V00315 (Inadequate Sewage Disposal): Preserved to cover effluent discharge into rudimentary cesspits, open ditches, and bodies of water.

V00316 (Absolute Absence of Bathroom or Toilet): Retained as the ultimate indicator of sanitation deprivation. The correlation test demonstrated that V00316 exhibited behavior perfectly opposite to that of the variables regarding exclusive bathrooms, justifying the exclusion of the latter in order to focus the model on severe deprivation.

7. Allocation Improper Waste Disposal:

V00399 to V00402 (Waste Burned, Buried, Dumped, or Otherwise Disposed Of): Retained to capture private households outside the regular collection service, whose final waste disposal practices exacerbate local socio-environmental burdens.

In [ ]:
import pandas as pd

# 1. List of deprivation variables (IBGE)
pre_selected_variables = [
    "V00005",
    "V00006",
    "V00008",
    "V00009",
    "V00050",
    "V00051",
    "V00052",
    "V00053",
    "V00054",
    "V00055",
    "V00056",
    "V00057",
    "V00058",
    "V00113",
    "V00114",
    "V00115",
    "V00116",
    "V00117",
    "V00118",
    "V00200",
    "V00201",
    "V00464",
    "V00236",
    "V00237",
    "V00312",
    "V00313",
    "V00314",
    "V00315",
    "V00316",
    "V00399",
    "V00400",
    "V00401",
    "V00402",
]

# 2. Generates, filters, renames, and rounds the descriptive statistics
df_descriptive = (
    df[pre_selected_variables]
    .describe()
    .T[["count", "mean", "std", "min", "max"]]
    .rename(
        columns={
            "count": "Count",
            "mean": "Mean",
            "std": "Standard Deviation",
            "min": "Minimum",
            "max": "Maximum",
        }
    )
    .round(2)
)

# 3. Saves the header and the table directly to a local TXT file
txt_output = "descriptive_statistics_pre_selection_table.txt"
with open(txt_output, "w", encoding="utf-8") as f:
    f.write(
        "DESCRIPTIVE STATISTICS TABLE (RAW VARIABLES) - PRE-SELECTED.\n\n"
    )
    f.write(df_descriptive.to_string())

# 4. Displays results
display(df_descriptive)

An analysis of the descriptive statistics for the 33 raw deprivation variables reveals the socio-spatial heterogeneity characterizing the territory of São Paulo state. The data range is vast: while the municipal resident population (V00005) starts at a minimum of 907 inhabitants, the demographic peak reaches over 11.3 million individuals. This disparity in scale is directly reflected in the standard deviation of most absolute deprivation indicators, where values ​​exceed their respective means by significant orders of magnitude.

Variables associated with critical urban precariousness—such as the total number of households in tenements (*cortiços*, V00050)—show minimum values ​​of zero, suggesting the absence of these phenomena in a large number of small municipalities. Conversely, maximum values ​​reach substantial figures (29,855 households), highlighting a strong concentration of extreme housing vulnerability within the state's metropolitan regions and major economic hubs.

Furthermore, the assessment identified that variable V00051 showed a mean, standard deviation, minimum, and maximum of zero across the entire sample set (645 municipalities). Due to the total absence of statistical variance, this metric was deemed irrelevant for predictive modeling or clustering purposes, justifying its discontinuation and exclusion prior to the feature engineering stages.

3.5 Methodological Isolation of Denominators and Descriptive Analysis

To conduct the descriptive statistical analysis and generate the heat map, the variables V00001 (Occupied Permanent Private Households – DPPO) and v0007 (Total Occupied Private Households – DPPO + DPIO) were intentionally isolated and excluded from the correlation tests. Because they function strictly as demographic denominators during the feature engineering stage, these columns measure only the absolute size of the municipalities rather than their structural deficits. Including them at this stage would distort the vulnerability analysis, as their high correlation with all other variables is a logical consequence of population size.

Consequently, the descriptive assessment focused exclusively on the 33 variables representing raw data on absolute deprivation and social profiles. The results confirmed the significant heterogeneity among municipalities in São Paulo state. Minimum and maximum values ​​reveal drastically different orders of magnitude between infrastructure variables and those related to extreme housing vulnerability (such as improvised structures in public spaces); the latter show low averages in most cities but severe peaks concentrated in large urban centers.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 1. Filters the variable list removing V00051
filtered_variables = [v for v in pre_selected_variables if v != "V00051"]

# 2. Calculates the correlation matrix
correlation_matrix = df[filtered_variables].corr(method="pearson")

# 3. Extracts, sorts, and formats the correlation pairs
df_pairs = (
    correlation_matrix.where(
        np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
    )
    .stack()
    .reset_index()
)
df_pairs.columns = ["Variable A", "Variable B", "Correlation"]
df_pairs = df_pairs.sort_values(by="Correlation", ascending=False)

# 4. Saves results directly to a local TXT file
txt_output = "raw_variables_correlation_list.txt"
with open(txt_output, "w", encoding="utf-8") as f:
    f.write("=== RAW VARIABLES CORRELATION LIST ===\n")
    f.write("Note: Variable V00051 was removed due to lack of variance.\n\n")
    f.write(
        df_pairs.to_string(
            index=False, formatters={"Correlation": "{:.4f}".format}
        )
    )

print(f"TXT file saved to: {txt_output}")

# 5. Configures and generates the Heatmap
plt.figure(figsize=(32, 26))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(
    correlation_matrix,
    mask=mask,
    cmap="coolwarm",
    vmax=1.0,
    vmin=-1.0,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.7, "label": "Pearson Correlation Coefficient"},
)

plt.title(
    "Pearson Correlation Matrix - Raw Census Variables",
    fontsize=24,
    fontweight="bold",
    pad=20,
)
plt.xticks(rotation=45, ha="right", fontsize=12)
plt.yticks(rotation=0, fontsize=12)
plt.tight_layout()

# 6. Saves and displays the plot locally
plot_output = "raw_variables_correlation_heatmap.png"
plt.savefig(plot_output, dpi=300, bbox_inches="tight")
print(f"Plot successfully saved to: {plot_output}")

plt.show()

3.6 The Demographic Scale Phenomenon: Why Raw Data Obscure Reality?

A detailed mathematical analysis of the correlation matrix for raw variables revealed a statistical pattern demanding extreme methodological caution. As evidenced by the list of Pearson coefficients, nearly all absolute variables regarding urban deprivation exhibit extraordinarily high linear associations, with values ​​frequently exceeding the $0.95$ threshold.

At first glance, a cursory reading might suggest that variables such as the total number of households lacking a private bathroom, the population living in slums, and the volume of substandard housing share an absolute conceptual redundancy. However, scientific analysis points to a classic statistical bias in population studies: the Demographic Scale Effect.

In practical terms, large municipalities (such as the metropolises of São Paulo state) record—by sheer mathematical logic—massive absolute volumes across all census counts simultaneously. A large metropolis will concentrate more households, more children, more sewage networks, and, inevitably, a higher raw number of tenements and substandard dwellings. Conversely, municipalities with small populations will record counts close to zero across virtually all economic and structural indicators.

When these population extremes are cross-referenced using absolute data, the correlation coefficient ceases to measure the actual intensity or social nature of vulnerability and instead measures solely the municipality's population size. The graphical result of this phenomenon is a heat map homogeneously saturated with shades indicating strong positive correlation (a "sea of ​​blue"), which masks the territory's true dynamics of deprivation.

Empirical evidence for this diagnosis emerges when analyzing the lower end of the correlation list, where the scale effect loses its dominance and structural geographic disparities come to light. Variables such as the presence of *cortiços* (tenements) and extreme urban poverty showed a near-zero (and sometimes negative) correlation when cross-referenced with sewage disposal via rudimentary septic tanks. This phenomenon occurs because rudimentary septic tanks are a predominant feature in rural areas or small inland towns, whereas tenements and high-density *favelas* are typically metropolitan phenomena. When population scale is weighed against the actual geographic distribution of the issue, the illusion of perfect correlations dissolves.

Feeding these raw absolute figures directly into Artificial Intelligence algorithms or machine learning clustering models would yield scientifically invalid results. The mathematical models would be misled, merely classifying municipalities into a linear ranking of "Large Cities vs. Small Cities" while failing to identify severe pockets of proportional destitution in smaller municipalities. Therefore, to neutralize the influence of population size and ensure a fair, symmetrical, and scientifically rigorous basis for comparing heterogeneous territories, transitioning to the feature engineering phase becomes mathematically essential. In this subsequent stage, isolated demographic denominators (V00001 and v0007) will be used to normalize the raw counts, transforming absolute volumes into proportional rates of deprivation per household and per inhabitant.

In [ ]:
import pandas as pd

# 1. List of household variables
household_variables = [
    "V00050",
    "V00052",
    "V00053",
    "V00054",
    "V00055",
    "V00056",
    "V00057",
    "V00058",
    "V00113",
    "V00114",
    "V00115",
    "V00116",
    "V00117",
    "V00118",
    "V00200",
    "V00201",
    "V00236",
    "V00237",
    "V00312",
    "V00313",
    "V00314",
    "V00315",
    "V00316",
    "V00399",
    "V00400",
    "V00401",
    "V00402",
    "V00464",
]

# 2. Calculates all household rates simultaneously (households / total occupied households)
df_rates = df[household_variables].div(df["v0007"], axis=0) * 100
df_rates.columns = [f"{var}_P" for var in household_variables]

# 3. Person-based rates use person denominators (people / people)
# V00006_P: residents in improvised dwellings / total residents (permanent + improvised)
# V00009_P: children 0-9 in improvised dwellings / total children 0-9 (permanent + improvised)
df_rates["V00006_P"] = df["V00006"] / (df["V00005"] + df["V00006"]) * 100
df_rates["V00009_P"] = df["V00009"] / (df["V00008"] + df["V00009"]) * 100

# 4. Inserts identifiers at the beginning of the DataFrame
df_rates.insert(0, "CD_MUN", df["CD_MUN"].values)
df_rates.insert(1, "NM_MUN", df["NM_MUN"].values)

if "NM_UF" in df.columns:
    df_rates.insert(2, "NM_UF", df["NM_UF"].values)
if "CD_UF" in df.columns:
    df_rates.insert(3, "CD_UF", df["CD_UF"].values)

# 5. Generates the descriptive statistics table
rates_descriptive = (
    df_rates.describe()
    .T[["count", "mean", "std", "min", "max"]]
    .rename(
        columns={
            "count": "Count",
            "mean": "Mean (%)",
            "std": "Standard Deviation",
            "min": "Minimum (%)",
            "max": "Maximum (%)",
        }
    )
    .round(2)
)

# 6. Saves the descriptive statistics table directly to a local TXT file
txt_output = "selected_variables_rates_statistics_table.txt"
with open(txt_output, "w", encoding="utf-8") as f:
    f.write(
        "DESCRIPTIVE STATISTICS TABLE (PROPORTIONS WITH UNIFIED DENOMINADOR v0007)\n"
    )
    f.write(
        "Household variables deflated by Total Occupied Households (v0007); V00006 and V00009 by their person totals.\n\n"
    )
    f.write(rates_descriptive.to_string())

# 7. Displays results
display(rates_descriptive)

After converting the raw data into percentages, the new descriptive table reveals a much more balanced and realistic picture of vulnerability in the state of São Paulo. By using the Total Occupied Households as the sole denominator for housing variables, the effect of city size was eliminated, allowing for a fair comparison between small and large municipalities.

The results indicate that, although the average rate for most forms of urban deprivation is low (often below 1%), there are municipalities where the situation is critical. The V00464_P indicator, for instance, shows a state average of 10.52% but reaches an alarming peak of 74.17% in a single locality. Similar patterns of intense local concentration appear in variables V00312_P (maximum of 55.73%) and V00114_P (maximum of 47.46%), highlighting severe pockets of substandard conditions that were previously masked by absolute figures.

Additionally, the analysis revealed that variable V00058_P consistently showed a value of zero across all statistics (mean, minimum, and maximum of 0.00%). Due to the complete lack of variation across the state of São Paulo, this metric was deemed irrelevant; it will be excluded from the dataset prior to training the Artificial Intelligence models, alongside the previously discontinued variable.

Analyzing the unified percentage rates relative to the total number of occupied households (v0007) made it possible to identify the true scale of housing deprivation in the state of São Paulo, free from distortions caused by city size. Overall, state averages for most extreme vulnerabilities are low, indicating that scenarios of severe housing precariousness are not the norm across the state's territory.

However, the maximum values ​​reveal profound regional inequality and raise critical red flags for public policy. The most severe case involves variable V00464_P: while the state average for households lacking a water connection is 10.52%, one municipality shows an alarming rate of 74.17%. Severe inequalities were also found regarding sanitation; one city recorded 55.73% of households discharging sewage into rudimentary cesspits (V00312_P), while another showed 47.46% reliance on shallow wells or open pits (V00114_P).

Finally, the data confirmed that variable V00058_P remained at zero across the entire state. Due to this complete lack of variation, it will be discarded to optimize the dataset used to train the artificial intelligence algorithms. Conversely, variable V00056_P—despite showing very low values ​​(with a maximum of just 0.19%)—will be retained in the study. Although statistically small, this metric serves as an indicator of a rare event (a social anomaly), carrying significant qualitative weight that will help the model identify municipalities with very specific scenarios of extreme street-level vulnerability.

# 📑 Dictionary of Selected Variables — SDG Target 11.1 (2022 Census)

This dictionary presents the final selection of variables from the 2022 Demographic Census (IBGE) used in the urban vulnerability modeling. The methodological strategy consisted of isolating demographic denominators and retaining **only variables indicative of deficit, deprivation, or inadequacy**, ensuring a unique and consistent directionality for Principal Component Analysis (PCA) and Clustering ($K$-Means).

---

### 1. Control Variables and Denominators (Methodological Weights)
* **`CD_MUN`**: Official municipality identification code (7 digits - IBGE).
* **`V00001`**: Occupied Permanent Private Households *(Physical Infrastructure Denominator)*.
* **`V00007`**: Total Occupied Private Households (Permanent + Improvised) *(Absolute Vulnerability Denominator)*.
* **`V00005`**: Number of residents in Occupied Permanent Private Households.
* **`V00006`**: Number of residents in Occupied Improvised Private Households.
* **`V00008`**: Number of children aged 0 to 9 in Occupied Permanent Private Households.
* **`V00009`**: Number of children aged 0 to 9 in Occupied Improvised Private Households.
* *(`N_POP_DPPO`, resident population in favelas and urban communities, was removed from the main dataset and is used only as a supplementary variable.)*

---

### 2. Raw Housing and Demographic Inadequacy Variables
* **`V00050`**: Housing in a tenement or boarding house room.
* **`V00051`**: Indigenous housing without walls or communal hut (*maloca*).
* **`V00052`**: Degraded or unfinished permanent residential structure.
* **`V00053`**: Improvised household: tent or shack made of canvas, plastic, or fabric.
* **`V00054`**: Improvised household: inside an operating commercial or industrial establishment.
* **`V00055`**: Improvised household: natural shelter and other improvised structures.
* **`V00056`**: Improvised household: improvised structure in a public space (excluding tents/shacks).
* **`V00057`**: Improvised household: degraded or unfinished permanent non-residential structure.
* **`V00058`**: Improvised household: vehicle (car, truck, trailer, boat, etc.).

---

### 3. Raw Water Inadequacy Variables
* **`V00113`**: Water supply from a shallow well, water table, or hand-dug well (*cacimba*).
* **`V00114`**: Water supply from a spring or fountain.
* **`V00115`**: Water supply from a water tanker truck.
* **`V00116`**: Water supply from stored rainwater.
* **`V00117`**: Water supply from rivers, reservoirs, streams, lakes, and channels (*igarapés*).
* **`V00118`**: Other precarious form of water supply.
* **`V00200`**: Piped water is available, but only to the land/yard.
* **`V00201`**: Piped water does not reach the household at all.
* **`V00463`**: Connected to the general water supply network, but primarily uses another source *(Intermittency)*.
* **`V00464`**: Household has no connection to the general water distribution network *(Exclusion)*.

---

### 4. Raw Sanitation Inadequacy Variables (Sewage and Restrooms)
* **`V00236`**: Only a restroom shared by more than one household *(Tenements/Collective housing)*.
* **`V00237`**: Only a toilet pit or hole for waste (including those located on the land).
* **`V00312`**: Sewage disposal into a rudimentary cesspool or pit.
* **`V00313`**: Sewage disposal into an open ditch.
* **`V00314`**: Sewage disposal into a river, lake, stream, or sea (direct discharge).
* **`V00315`**: Other inadequate form of sewage disposal.
* **`V00316`**: Non-existent sewage disposal due to lack of restroom or toilet facilities *(Extreme Deprivation)*.

---

### 5. Raw Waste Management Variables (Garbage)
* **`V00399`**: Garbage burned on the property.
* **`V00400`**: Garbage buried on the property.
* **`V00401`**: Garbage dumped in a vacant lot, hillside, or public area.
* **`V00402`**: Other inadequate form of garbage disposal.

---

> 💡 **Methodological Note on Exclusion:**
> * Water comfort and socioeconomic stratification variables (**V00232 to V00235** and **V00485 to V00494** - count of multiple restrooms) were eliminated because they do not represent deprivation.
> * Full service and urban coverage variables (**V00111, V00112, V00199, V00309, V00310, V00311, V00397, and V00398**) were discarded to suppress the statistical noise of reverse multicollinearity (*mirror effect*) in the covariance matrix.
> * Variable **V00238** (absence of a restroom) was removed due to perfect multicollinearity and mathematical tautology with variable **V00316** (non-existent disposal due to lack of restroom), choosing to retain the latter in the corresponding block.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 1. Selects columns ending with '_P' and removes 'V00058_P'
final_rates = [
    t for t in df_rates.columns if t.endswith("_P") and t != "V00058_P"
]

# 2. Calculates the new correlation matrix for rates
rates_correlation_matrix = df_rates[final_rates].corr(method="pearson")

# 3. Extracts and sorts correlation pairs
df_pairs = (
    rates_correlation_matrix.where(
        np.triu(np.ones(rates_correlation_matrix.shape), k=1).astype(bool)
    )
    .stack()
    .reset_index()
)
df_pairs.columns = ["Variable A", "Variable B", "Correlation"]
df_pairs = df_pairs.sort_values(by="Correlation", ascending=False)

# 4. Saves the correlation list directly to a local TXT file
txt_output = "final_rates_correlation_list.txt"
with open(txt_output, "w", encoding="utf-8") as f:
    f.write("CORRELATION LIST FOR PERCENTAGE RATES (v0007)\n")
    f.write(
        "Analysis of real relationships after removing the demographic scale effect.\n\n"
    )
    f.write(
        df_pairs.to_string(
            index=False, formatters={"Correlation": "{:.4f}".format}
        )
    )

print(f"TXT file successfully saved to: {txt_output}")

# 5. Configures and generates the Heatmap
plt.figure(figsize=(30, 24))
mask = np.triu(np.ones_like(rates_correlation_matrix, dtype=bool))

sns.heatmap(
    rates_correlation_matrix,
    mask=mask,
    cmap="coolwarm",
    vmax=1.0,
    vmin=-1.0,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.7, "label": "Pearson Correlation Coefficient"},
)

plt.title(
    "Correlation Matrix - Unified Percentage Rates (v0007)",
    fontsize=22,
    fontweight="bold",
    pad=20,
)
plt.xticks(rotation=45, ha="right", fontsize=11)
plt.yticks(rotation=0, fontsize=11)
plt.tight_layout()

# 6. Saves and displays the plot locally
plot_output = "final_rates_correlation_heatmap.png"
plt.savefig(plot_output, dpi=300, bbox_inches="tight")
print(f"Plot successfully saved to: {plot_output}")

plt.show()

Redundancy Check on the Final Rates

The Pearson screening (|r| ≥ 0.85) was first applied to the raw counts, where the demographic scale effect inflates almost every correlation. The check is therefore repeated on the percentage rates, which reflect the real relationships between deprivations. For each pair with |r| ≥ 0.85, only one variable should be kept; the removed ones are listed in `redundant_to_remove` and justified in the text.

In [ ]:
import numpy as np

# 1. Pairs of rates with |r| >= 0.85 (same threshold used in the initial screening)
threshold = 0.85
corr_rates = df_rates[final_rates].corr(method="pearson")
pairs_rates = (
    corr_rates.where(np.triu(np.ones(corr_rates.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
pairs_rates.columns = ["Variable A", "Variable B", "Correlation"]
redundant_pairs = pairs_rates[pairs_rates["Correlation"].abs() >= threshold]
display(redundant_pairs.sort_values("Correlation", ascending=False).round(4))

# 2. DEFINE here which variables to remove after inspecting the pairs above
# (e.g., ["V00053_P"] if V00006_P and V00053_P are redundant). Leave empty to keep all.
redundant_to_remove = []
print(f"Variables removed for redundancy: {redundant_to_remove}")

The transition from the correlation matrix of raw variables to the matrix of percentage rates revealed a significant structural shift in data behavior. While the matrix of raw selected data displayed artificially high coefficients—driven by the demographic scale of the municipalities—the matrix generated from proportions (using the denominator v0007: Total Occupied Households) showed a predominantly neutral profile, with values ​​close to zero. This overall disconnect indicates that housing and infrastructure deprivations in the state of São Paulo do not occur homogeneously or linearly across the territory. However, the few linear correlations that remained significant reveal the true axes of associated vulnerability—where different dimensions of urban poverty structurally coexist within the same municipality—offering valuable insights to be explored by clustering algorithms and artificial intelligence.

> ⚠️ **To be updated after re-running** (the favela variable was removed and the denominators of V00006_P and V00009_P were corrected).

An analysis of the coefficients that survived demographic filtering revealed clear, inseparable axes of structural vulnerability. The most critical cluster relates to extreme housing precarity and homelessness, evidenced by the very strong correlation between the proportion of residents in makeshift dwellings (V00006_P) and the presence of tents or shacks in public spaces (V00053_P, r = 0.9284). This scenario is exacerbated by the strong association of these indicators with the number of children aged zero to nine living in such conditions (V00009_P, r = 0.8206), delineating areas of severe social exclusion.

A second structural axis emerges regarding basic sanitation and network infrastructure. A strong convergence is observed between the lack of connection to the public water supply network (V00464_P) and the disposal of sewage into rudimentary septic tanks or open pits (V00312_P, r = 0.7921)—a pattern also directly linked to a reliance on shallow wells or dug wells (V00114_P, r = 0.7063). This pattern indicates a historical infrastructure deficit that tends to be concentrated within the same municipalities.


In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Filter columns ending with "_P" directly from the source dataframe
rate_columns = [
    c for c in df_rates.columns
    if c.endswith("_P") and c not in ["V00058_P"] + redundant_to_remove
]
print(f"Variables used in the clustering: {len(rate_columns)}")

# 2. Normalize data using CD_MUN as a temporary index to keep rows aligned
df_rates_normalized = pd.DataFrame(
    MinMaxScaler().fit_transform(df_rates.set_index("CD_MUN")[rate_columns]),
    columns=rate_columns,
    index=df_rates["CD_MUN"]
).reset_index()

# 3. Save the normalized dataset locally and preview the output
df_rates_normalized.to_csv("df_rates_normalized.csv", index=False, encoding="utf-8")
display(df_rates_normalized.head())

It is crucial to highlight the methodological distinction and complementarity between data proportionalization and subsequent linear scaling normalization (Min-Max scaling). Applying the Min-Max algorithm directly to raw Census data would be both insufficient and incorrect; while normalization adjusts the geometric range of variables to an interval between 0 and 1, it preserves distortions caused by the municipalities' demographic scales. Without first dividing the data by the total number of occupied households (v0007), Artificial Intelligence algorithms would be biased by absolute population sizes—grouping municipalities based on population volume rather than the severity of their housing deprivations.

Therefore, data transformation was carried out in two mandatory, independent stages: first, proportionalization (division by the unified denominator v0007) was performed to eliminate the effect of city size and convert absolute counts into rates of deprivation intensity. Next, Min-Max normalization was applied to equalize the weight of different variables within the geometric clustering model, ensuring that indicators with historically low averages (such as extreme homelessness) held the same potential influence as indicators with higher prevalence (such as sanitation deficits). This two-tiered approach shields the model from biases related to demographic scale and variable magnitude, thereby ensuring the statistical accuracy of the clustering process.

The decision to apply the K-Means algorithm directly to the normalized rates was a deliberate methodological choice aimed at preserving the interpretability and analytical transparency of the results. Determining the optimal number of partitions (K) was supported by the mathematical criteria of the Elbow Method (Inertia) and the Silhouette Coefficient, ensuring the formation of clusters that are internally homogeneous and maximally distinct from one another.

Although PCA is effective for reducing data volume, it transforms the original Census variables into abstract orthogonal components; this would make it impossible to directly and individually identify which infrastructure deficits (such as the exact percentage of water shortages or inadequate sanitation) shape each cluster. Given that this study aims to inform urban planning and the targeting of specific public policies, retaining the variables in their original form ensures that the clusters generated by the Artificial Intelligence maintain immediate empirical alignment with the physical and social reality of the municipalities in São Paulo state, without any loss of sociological meaning.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Load data, setting CD_MUN as index to isolate it from KMeans features
df_filtered = pd.read_csv("df_rates_normalized.csv", index_col="CD_MUN")

# 1. Run K-Means and compute metrics
k_range = range(2, 11)
models = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(df_filtered) for k in k_range]

inertias = [model.inertia_ for model in models]
silhouette_scores = [silhouette_score(df_filtered, model.labels_) for model in models]

# 2. Configure dual-axis plot (Elbow vs. Silhouette)
fig, ax1 = plt.subplots(figsize=(10, 5))

# Inertia Plot (Left Axis - Blue)
ax1.set_xlabel("Number of Clusters (K)", fontsize=11, fontweight="bold")
ax1.set_ylabel("Inertia", color="tab:blue", fontsize=11)
ax1.plot(k_range, inertias, marker="o", color="tab:blue", linewidth=2)
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.grid(True, linestyle="--", alpha=0.5)

# Silhouette Score Plot (Right Axis - Red)
ax2 = ax1.twinx()
ax2.set_ylabel("Silhouette Coefficient", color="tab:red", fontsize=11)
ax2.plot(k_range, silhouette_scores, marker="s", color="tab:red", linewidth=2, linestyle="--")
ax2.tick_params(axis="y", labelcolor="tab:red")

plt.title("Cluster Selection: Elbow Method vs. Silhouette Score", fontsize=12, fontweight="bold", pad=10)
fig.tight_layout()

# 3. Save and display the plot
plt.savefig("elbow_silhouette_selection_plot.png", dpi=300, bbox_inches="tight")
plt.show()

# 4. Display numerical results
display(pd.DataFrame({"K": k_range, "Inertia": inertias, "Silhouette": silhouette_scores}).round(4))

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans

# 1. Load normalized data, isolating CD_MUN from features by setting it as index
df_filtered = pd.read_csv("df_rates_normalized.csv", index_col="CD_MUN")

# 2. Instantiate and fit the final K-Means model (adjust n_clusters as needed)
optimal_k = 4  # DEFINE after reviewing the new Elbow/Silhouette results above (and justify the choice in the text)
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(df_filtered)

# 3. Append clusters to the dataset and bring CD_MUN back as a regular column
df_final = df_filtered.copy()
df_final["cluster"] = clusters
df_final = df_final.reset_index()

# 4. Save the final clustered dataset locally and preview results
df_final.to_csv("df_census_clusters.csv", index=False, encoding="utf-8")
display(df_final.head())

Cluster Stability and Robustness

Four checks to show that the clusters reflect a real structure in the data and do not depend on chance or on the chosen algorithm:

1. **Seeds:** K-Means is re-run with 50 different random seeds and each result is compared with the final model using the Adjusted Rand Index (ARI: 1 = identical groups; ~0 = agreement expected by chance). The comparison is based on which municipalities stay together, so the cluster numbers do not need to match.
2. **Cluster sizes:** checks whether any cluster contains only one or two municipalities.
3. **Silhouette by cluster:** shows how well-defined each cluster is, not only the overall average.
4. **Ward's hierarchical clustering:** a different, deterministic method with the same criterion (minimizing within-group variance). Agreement with K-Means (ARI and cross-tabulation) indicates that the typologies are in the data and not in the method. The dendrogram shows how municipalities merge step by step.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_samples

X = df_filtered.values

# 1. Stability across seeds
seed_labels = [
    KMeans(n_clusters=optimal_k, random_state=s, n_init=10).fit_predict(X)
    for s in range(50)
]
ari_seeds = np.array([adjusted_rand_score(clusters, lab) for lab in seed_labels])
print(f"1. Seeds - ARI vs. final model: mean {ari_seeds.mean():.3f} | "
      f"min {ari_seeds.min():.3f} | runs identical (ARI = 1): {(ari_seeds == 1).sum()} of 50")

# 2. Cluster sizes
print("\n2. Municipalities per cluster:")
print(pd.Series(clusters).value_counts().sort_index().to_string())

# 3. Silhouette by cluster
sil = silhouette_samples(X, clusters)
sil_by_cluster = pd.DataFrame({"cluster": clusters, "silhouette": sil}).groupby("cluster")["silhouette"].mean().round(3)
print(f"\n3. Mean silhouette by cluster (overall: {sil.mean():.3f}):")
print(sil_by_cluster.to_string())

# 4. Ward's hierarchical clustering
ward_labels = AgglomerativeClustering(n_clusters=optimal_k, linkage="ward").fit_predict(X)
print(f"\n4. Ward vs. K-Means - ARI: {adjusted_rand_score(clusters, ward_labels):.3f}")
crosstab = pd.crosstab(pd.Series(clusters, name="K-Means"), pd.Series(ward_labels, name="Ward"))
display(crosstab)

# Dendrogram (labels with municipality names)
names = df.set_index("CD_MUN").loc[df_filtered.index, "NM_MUN"].astype(str).values
plt.figure(figsize=(14, 6))
dendrogram(linkage(X, method="ward"), labels=names, leaf_rotation=90, leaf_font_size=9)
plt.title("Ward's Hierarchical Clustering - Dendrogram", fontsize=13, fontweight="bold")
plt.ylabel("Distance (Ward)")
plt.tight_layout()
plt.savefig("ward_dendrogram.png", dpi=300, bbox_inches="tight")
plt.show()

# Save a summary table
pd.DataFrame({
    "Check": ["ARI seeds (mean)", "ARI seeds (min)", "Silhouette (overall)", "ARI Ward vs K-Means"],
    "Value": [ari_seeds.mean(), ari_seeds.min(), sil.mean(), adjusted_rand_score(clusters, ward_labels)],
}).round(3).to_csv("stability_summary.csv", index=False, encoding="utf-8")

Cluster Visualization & Automated Feature Selection

To visualize the K-Means clustering results (final K) in a 2D space, an automated feature selection approach was implemented. Instead of manually picking arbitrary variables, the script evaluates all census proportions (`_P`) and calculates the **variance of the means** across the clusters for each feature.

The two features displaying the highest variance were automatically selected as the X and Y axes:
* **X-Axis:** The feature with the highest variance (the strongest differentiator among the groups).
* **Y-Axis:** The feature with the second-highest variance.

This method ensures that the resulting scatter plot captures the maximum possible visual separation and contrast, effectively highlighting the distinct socioeconomic profiles found by the algorithm.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calculate the variance of means across the clusters for each feature
features = [c for c in df_final.columns if c.endswith("_P")]
cluster_means = df_final.groupby("cluster")[features].mean()
mean_variances = cluster_means.var()

# 2. Automatically select the top 2 features with the highest variance across clusters
top_features = mean_variances.nlargest(2).index.tolist()
x_var, y_var = top_features[0], top_features[1]

print(f"Automatically selected features for maximum cluster separation:")
print(f"X-Axis: {x_var} (Variance: {mean_variances[x_var]:.4f})")
print(f"Y-Axis: {y_var} (Variance: {mean_variances[y_var]:.4f})")

# 3. Configure the 2D scatter plot for all clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_final,
    x=x_var,
    y=y_var,
    hue="cluster",
    palette="Set1",
    alpha=0.8,
    s=50,
)

# 4. Add titles, labels, and formatting
plt.title("Automated K-Means Visualization", fontsize=13, fontweight="bold", pad=15)
plt.xlabel(f"{x_var} (Highest Variance Across Clusters)", fontsize=11)
plt.ylabel(f"{y_var} (Second Highest Variance Across Clusters)", fontsize=11)
plt.legend(title="Cluster", loc="best")
plt.grid(True, linestyle="--", alpha=0.5)

# 5. Save and display the plot locally
plt.savefig("kmeans_auto_2d_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import plotly.express as px

# 1. Calculate the variance of means across the clusters for each feature
features = [c for c in df_final.columns if c.endswith("_P")]
cluster_means = df_final.groupby("cluster")[features].mean()
mean_variances = cluster_means.var()

# 2. Automatically select the top 3 features with the highest variance
top_features = mean_variances.nlargest(3).index.tolist()
var_x, var_y, var_z = top_features[0], top_features[1], top_features[2]

print(f"Automatically selected for Thesis Plot -> X: {var_x} | Y: {var_y} | Z: {var_z}")

# 3. Prepare the dataframe matching your exact naming convention
df_plot_3d = df_final.copy()
df_plot_3d["Cluster"] = df_plot_3d["cluster"].astype(str)  # Categorical mapping for Set2

# 4. Generates the interactive 3D scatter plot using your custom layout
fig = px.scatter_3d(
    df_plot_3d,
    x=var_x,
    y=var_y,
    z=var_z,
    color="Cluster",
    color_discrete_sequence=px.colors.qualitative.Set2,
    title=f"3D Spatial Visualization of SP Municipalities",
    hover_data=["CD_MUN"],  # Keeps the municipality identifier on hover
)

# Esthetic adjustments for thesis quality (Your exact template)
fig.update_traces(
    marker=dict(size=4, opacity=0.7, line=dict(width=0.5, color="White"))
)
fig.update_layout(
    margin=dict(l=0, r=0, b=0, t=40),
    width=900,
    height=700,
    legend=dict(title="Clusters"),
)

# 5. Display the interactive plot and save it
fig.show()
fig.write_html("kmeans_3d_plot.html")

In [ ]:
#!pip install geobr geopandas

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# 1. Download official municipality shapefile directly from IBGE
sp_mesh = gpd.read_file(
    "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/UFs/SP/SP_Municipios_2022.zip"
)

# 2. Ensure both municipality code columns match exactly in type (int)
# IBGE's 2022 mesh uses 'CD_MUN' as the standard column name
sp_mesh["CD_MUN"] = sp_mesh["CD_MUN"].astype(int)
df_final["CD_MUN"] = df_final["CD_MUN"].astype(int)

# 3. Merge the geographic mesh with your final cluster results
gdf_map = sp_mesh.merge(df_final, on="CD_MUN", how="inner")

# 4. Configure and plot the map
fig, ax = plt.subplots(figsize=(12, 8))

# Plot the background in light gray (helps identify if any municipality was left out)
sp_mesh.plot(ax=ax, color="#E0E0E0", edgecolor="white", linewidth=0.3)

# Plot the clustered municipalities using Set2 to match your 3D plot aesthetics
gdf_map.plot(
    column="cluster",
    categorical=True,
    legend=True,
    ax=ax,
    cmap="Set2",
    edgecolor="white",
    linewidth=0.4,
    legend_kwds={"title": "Clusters", "loc": "lower right", "fmt": "{:.0f}"}
)

# 5. Thesis-quality esthetic adjustments
ax.set_title("Spatial Distribution of Census Clusters in São Paulo State", fontsize=14, fontweight="bold", pad=15)
ax.set_axis_off()  # Removes latitude/longitude bounding box
fig.tight_layout()

# 6. Save and display the map locally
output_map = "census_clusters_spatial_map.png"
plt.savefig(output_map, dpi=300, bbox_inches="tight")
print(f" Map successfully generated and saved to: {output_map}")
plt.show()

In [ ]:
import pandas as pd

# PANDAS CONFIGURATION: Force full display without truncating rows
pd.set_option("display.max_rows", None)

# 1. Load the local cluster results from your actual previous cell output
df_results = pd.read_csv("df_census_clusters.csv")

# 2. Automatically select rate columns (ending in _P) and the cluster column
rate_columns = [c for c in df_results.columns if c.endswith("_P")]
df_rates_cluster = df_results[rate_columns + ["cluster"]]

# 3. DYNAMIC CENTROID PROFILE CONSTRUCTION (Adapts automatically to any K)
df_centroids_real = df_rates_cluster.groupby("cluster").mean().T

# Automatically generate column names based on the chosen optimal_k (e.g., Cluster_0_Mean, Cluster_1_Mean...)
mean_columns = [f"Cluster_{i}_Mean" for i in range(optimal_k)]
df_centroids_real.columns = mean_columns

# 4. MULTICLUSTER SORTING CRITERION (Intergroup Disparity)
# Calculates standard deviation across all group means dynamically
df_centroids_real["Intergroup_Disparity"] = df_centroids_real[mean_columns].std(axis=1)

# Sort the report in descending order based on the magnitude of disparity
df_centroids_real = df_centroids_real.sort_values(
    by="Intergroup_Disparity", ascending=False
)

# 5. Save the final centroids analysis dataset locally with a dynamic filename
output_centroids_csv = f"centroids_analysis_{optimal_k}_clusters.csv"
df_centroids_real.to_csv(output_centroids_csv, encoding="utf-8")
print(f"✅ Centroid profiles analysis saved as '{output_centroids_csv}'")

# 6. TRANSLATION DICTIONARY (International Demographic Standard)
legend_dict_en = {
    "V00001_P": "OPPD - Proportion of Occupied Permanent Private Dwellings",
    "v0007_P": "Total Occupied Private Dwellings (OPPD + OPID)",
    "V00005_P": "OPPD - Average number of residents",
    "V00008_P": "OPPD - Proportion of children aged 0 to 9",
    "V00006_P": "OPID - Residents in improvised dwellings (% of total residents)",
    "V00009_P": "OPID - Children aged 0 to 9 in improvised dwellings (% of total children 0 to 9)",
    "V00050_P": "OPPD Type - Rooming house or tenement (cortiço)",
    "V00051_P": "OPPD Type - Indigenous housing without walls (maloca)",
    "V00052_P": "OPPD Type - Degraded or unfinished residential structure",
    "V00053_P": "OPID Type - Tent or canvas/plastic/fabric shack",
    "V00054_P": "OPID Type - Located within an operating commercial establishment",
    "V00055_P": "OPID Type - Natural shelter and other structures",
    "V00056_P": "OPID Type - Improvised structure in public spaces/streets",
    "V00057_P": "OPID Type - Degraded or unfinished non-residential structure",
    "V00058_P": "OPID Type - Vehicle (car, truck, trailer, boat, etc.)",
    "V00113_P": "OPPD Water - Uses shallow well, water table, or cistern",
    "V00114_P": "OPPD Water - Uses natural spring, source, or mine",
    "V00115_P": "OPPD Water - Uses water tank truck (carro-pipa)",
    "V00116_P": "OPPD Water - Uses stored rainwater",
    "V00117_P": "OPPD Water - Uses rivers, lakes, streams, or ponds",
    "V00118_P": "OPPD Water - Uses other forms of water supply",
    "V00200_P": "OPPD Water - Piped water, but only to the property/land",
    "V00201_P": "OPPD Water - No piped water supply directly to the dwelling",
    "V00236_P": "OPPD Bathroom - Shared use bathroom/toilet facility",
    "V00237_P": "OPPD Bathroom - Basic pit latrine or makeshift toilet structure",
    "V00312_P": "OPPD Sewage - Rudimentary pit or cesspool",
    "V00313_P": "OPPD Sewage - Open ditch / Open-air sewage",
    "V00314_P": "OPPD Sewage - Discharged directly into rivers, lakes, streams, or sea",
    "V00315_P": "OPPD Sewage - Other forms of wastewater disposal",
    "V00316_P": "OPPD Sewage - Inexistent (no bathroom or toilet facility)",
    "V00399_P": "OPPD Waste - Burned within the property",
    "V00400_P": "OPPD Waste - Buried within the property",
    "V00401_P": "OPPD Waste - Disposed of in vacant lots, hillsides, or public areas",
    "V00402_P": "OPPD Waste - Other destination given to waste",
    "V00464_P": "OPPD Water - No connection to the general water distribution network",
}

# 7. FORMATTED TABLE DISPLAY
print(f"\n=== CENTROID PROFILES ({len(df_centroids_real)} Variables Translated and Sorted by Intergroup Disparity) ===")

# Apply English translation mapping and horizontal gradient
final_table_en = (
    df_centroids_real.rename(index=legend_dict_en)
    .round(4)
    .style.background_gradient(cmap="Blues", axis=1, subset=mean_columns)
)

# Render formatted table
display(final_table_en)

# Reset Pandas row display configuration to default
pd.reset_option("display.max_rows")

> ⚠️ **To be rewritten after re-running.** The interpretation below refers to the previous run (with the favela variable and the old denominators).

Cluster Profile Analysis and Interpretation

The K-Means clustering algorithm ($K=4$) partitioned the municipalities of São Paulo State into highly distinct socio-demographic and infrastructural archetypes. By sorting the variables using **Intergroup Disparity** (measured by the standard deviation of across-cluster normalized means), the analysis reveals that structural sanitation frameworks (sewage and water networks) and territorial informality are the primary drivers of spatial differentiation across the state.

#### **Cluster 0: Isolated Rural and Precarious Structural Vulnerability**
This cluster groups municipalities characterized by a severe deficit in centralized public utility networks, displaying a distinct rural-peripheral profile. It exhibits the state's highest reliance on rudimentary pits or cesspools ($\mu = 0.4826$) and the highest proportion of households lacking direct connection to the general water distribution network ($\mu = 0.4677$). Consequently, these territories rely heavily on localized alternatives, such as natural springs or source water ($\mu = 0.4159$) and shallow wells/cisterns ($\mu = 0.2199$). Waste management is similarly precarious, with high rates of burning trash within properties ($\mu = 0.1693$). This profile points toward an infrastructural gap driven by geographic isolation or lack of public utility expansion in rural regions.

#### **Cluster 1: Consolidated Urban Infrastructure (The Baseline)**
Cluster 1 represents the urbanized and structurally consolidated baseline of São Paulo State. It systematically presents the lowest normalized scores for every indicator of precarity. Deficits in the general water network ($\mu = 0.0796$) and the usage of rudimentary pits ($\mu = 0.0654$) are nominal. Critical vulnerabilities, such as informal settlements ($\mu = 0.0073$) or raw sewage discharge into water bodies ($\mu = 0.0135$), are virtually absent. This cluster encompasses municipalities where urban planning and basic infrastructure have successfully kept pace with demographic demands.

#### **Cluster 2: Metropolitan Informality and Environmental Risk**
Cluster 2 captures a highly specific urban archetype defined by acute socioeconomic pressure and territorial informality, typical of large metropolitan peripheries or industrialized hubs. This group dramatically outscores all others in the proportion of the population residing in **favelas and informal settlements ($\mu = 0.3857$)**, rooming houses/tenements ($\mu = 0.2096$), and degraded or unfinished residential structures ($\mu = 0.1622$). This intense spatial density is accompanied by severe environmental impacts, as evidenced by the highest rate of raw sewage discharged directly into rivers, lakes, and seas ($\mu = 0.2489$).

#### **Cluster 3: Transitional Semi-Rural Profile**
Cluster 3 reflects an intermediate archetype that bridges the gap between urban consolidation and rural isolation. Municipalities in this group share some structural deficits with Cluster 0, such as moderate rates of rudimentary pit usage ($\mu = 0.2538$) and lack of general water network coverage ($\mu = 0.2275$). However, they are distinct from Cluster 0 because they do not rely on natural springs/mines ($\mu = 0.0521$), indicating alternative regional setups. Waste management is a notable marker for this cluster, which presents the state's highest rate of domestic waste burning ($\mu = 0.1719$) and alternative waste disposal destinations ($\mu = 0.1445$).

Supplementary Variable: Population in Favelas and Urban Communities

The resident population in favelas and urban communities (N_POP_DPPO) did not take part in the clustering. It is analyzed here only as a supplementary variable: after the clusters are formed, its proportion is summarized by cluster and mapped as a separate layer, to support the interpretation of the typologies.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# 1. Load the supplementary file generated by the database notebook (BD_)
df_sup = pd.read_csv(path + "favelas_suplementar_SP.csv", sep=";")
df_sup["CD_MUN"] = df_sup["CD_MUN"].astype(int)

# 2. Proportion of residents in favelas (over residents in permanent dwellings, V00005)
base_pop = df[["CD_MUN", "V00005"]].copy()
base_pop["CD_MUN"] = base_pop["CD_MUN"].astype(int)
df_sup = df_sup.merge(base_pop, on="CD_MUN", how="left")
df_sup["FAVELA_P"] = df_sup["N_POP_DPPO"] / df_sup["V00005"] * 100

# 3. Join with the clusters (outside the K-Means) and summarize by cluster
df_sup = df_sup.merge(df_final[["CD_MUN", "cluster"]], on="CD_MUN", how="inner")
summary_sup = df_sup.groupby("cluster")["FAVELA_P"].agg(
    Municipalities="count",
    Mean="mean",
    Median="median",
    With_favelas=lambda x: (x > 0).sum(),
).round(2)
display(summary_sup)
summary_sup.to_csv("favelas_by_cluster.csv", encoding="utf-8")

# 4. Map: clusters (left) and favela population as a separate layer (right)
gdf_sup = sp_mesh.merge(df_sup, on="CD_MUN", how="inner")
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
gdf_map.plot(column="cluster", categorical=True, legend=True, cmap="Set2",
             edgecolor="white", linewidth=0.3, ax=axes[0],
             legend_kwds={"title": "Clusters", "loc": "lower right"})
axes[0].set_title("Census clusters", fontsize=13, fontweight="bold")
gdf_sup.plot(column="FAVELA_P", cmap="Reds", legend=True, edgecolor="white",
             linewidth=0.3, ax=axes[1],
             legend_kwds={"label": "% of residents in favelas", "shrink": 0.6})
axes[1].set_title("Supplementary layer: population in favelas (%)", fontsize=13, fontweight="bold")
for ax in axes:
    ax.set_axis_off()
fig.tight_layout()
plt.savefig("clusters_vs_favelas_map.png", dpi=300, bbox_inches="tight")
plt.show()